# ShimbaLLM — CROP training on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShimbaLLM/shimballm/blob/main/notebooks/ShimbaLLM_CROP_Colab.ipynb)

This notebook trains a small language model with the **CROP** recipe (Curriculum-guided Reduction Of Pretraining) from the paper *"Smaller, Faster, Smarter"* — no local GPU required:

1. **Phase 0** — a frozen reference model scores every document (excess loss).
2. **Stage A** — drop the bottom-30% documents, full loss, short context, WSD warmup→stable.
3. **Stage B** — selectivity: mask the easiest tokens (ρ 0.40→0.10), ramp context 2K→4K.
4. **Stage C** — anneal the LR on the top-quality slice → final model.
5. **Recycle** — re-anneal extra branches from the Stage-B stable checkpoint (~15% cost each).

**Setup:** `Runtime → Change runtime type → T4 GPU` (free tier works).
The notebook auto-detects the hardware and picks one of two tracks:

| Track | When | Config | Time |
|---|---|---|---|
| `cpu` | no GPU assigned | `examples/tiny_cpu.yaml` (synthetic corpus, ~0.3M params) | ~2 min |
| `gpu` | T4 / L4 / A100 | `examples/smollm_135m_hf.yaml` (SmolLM2-135M on FineWeb-Edu) | ~1–2 h at the default 30M-token budget |

Every long-running cell uses `--resume`, so a disconnected session continues from `last.pt` instead of starting over.

In [ ]:
import sys, torch

print('Python  :', sys.version.split()[0])
print('PyTorch :', torch.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU     : {p.name} | {p.total_memory/1e9:.1f} GB VRAM | '
          f'compute capability {p.major}.{p.minor}')
    TRACK = 'gpu'
else:
    print('GPU     : none -> CPU track (tiny smoke run)')
    TRACK = 'cpu'

## 1 · Install

Clone the repo and install the package. If the repository is still private,
upload the project ZIP to Colab and unzip it into `./shimballm` instead —
the install cell works either way.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/ShimbaLLM/shimballm.git'   # <- change to your fork if needed

if not os.path.isdir('shimballm'):
    r = subprocess.run(['git', 'clone', REPO_URL], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.strip())
        raise SystemExit(
            'Could not clone the repo. If it is still private, upload the '
            'project ZIP and unzip it so that ./shimballm exists, then re-run.')

%cd shimballm
!pip -q install -e ".[hf]"

import shimballm
print('shimballm', shimballm.__version__, 'ready')

## 2 · Data (GPU track only)

The real track trains **SmolLM2-135M** on a slice of [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) — the same open, reproducible data stack the paper's protocol section uses. We stream 20,000 documents to `data/fineweb-edu-sample.jsonl` (a few minutes on Colab; raise `N_DOCS` for a longer run).

In [ ]:
if TRACK == 'gpu':
    import json, os
    from datasets import load_dataset

    N_DOCS = 20_000
    ds = load_dataset('HuggingFaceFW/fineweb-edu', 'sample-10BT',
                      split='train', streaming=True)
    os.makedirs('data', exist_ok=True)
    n = 0
    with open('data/fineweb-edu-sample.jsonl', 'w') as f:
        for row in ds:
            f.write(json.dumps({'text': row['text']}) + '\n')
            n += 1
            if n >= N_DOCS:
                break
    print(f'wrote {n} documents -> data/fineweb-edu-sample.jsonl')
else:
    print('CPU track uses the built-in synthetic corpus — nothing to download.')

## 3 · Hardware-aware configuration

Writes `colab_run.yaml` from the template config, with three Colab-specific
patches on the GPU track:

* **AMP dtype** — a T4 (compute capability 7.x) has no bf16 support → `fp16`;
  Ampere+ (L4, A100) keeps `bf16`.
* **Micro-batch** — effective batch stays 64, split as 2 sequences × 4K ctx
  × 32 gradient-accumulation steps, so it fits in 16 GB VRAM.
* **Token budget** — 30M tokens ≈ 1–2 h on a free T4. Scale it up when you
  have more session time; the FLOPs projection below updates automatically.

In [ ]:
import yaml, torch

BASE = 'examples/smollm_135m_hf.yaml' if TRACK == 'gpu' else 'examples/tiny_cpu.yaml'
cfg = yaml.safe_load(open(BASE))

if TRACK == 'gpu':
    cc = torch.cuda.get_device_capability(0)
    cfg['schedule']['amp_dtype'] = 'bf16' if cc[0] >= 8 else 'fp16'
    cfg['schedule']['batch_size'] = 64
    cfg['schedule']['grad_accum'] = 32        # micro-batch = 64/32 = 2 seqs
    cfg['schedule']['total_tokens'] = 30_000_000
    cfg['engine']['eval_every'] = 250
    cfg['hub']['repo_name'] = 'crop-smollm2-135m-colab'

yaml.safe_dump(cfg, open('colab_run.yaml', 'w'), sort_keys=False)
OUT = cfg['engine']['out_dir']
print(f'track={TRACK}  base={BASE}  out_dir={OUT}')
print(open('colab_run.yaml').read())

## 4 · Project the compute bill first

Before spending any GPU-minutes, print the FLOPs accounting (paper Table 1 style) for *this* configuration: document gating, the context ramp and the reference-scoring overhead, versus a uniform-training baseline.

In [ ]:
!shimballm project --config colab_run.yaml

## 5 · Train (Phase 0 → Stage A → Stage B → Stage C)

One command runs the full Algorithm 1. Stage transitions are **gated by the validation excess-loss slope**, not fixed step counts, so stage lengths adapt to how noisy the corpus is. `--resume` re-enters from `last.pt` if the session restarts.

Artifacts land in `runs/…`: `final.pt`, `stable.pt` (the recycling asset),
`summary.json`, `log.jsonl` and `cache/doc_scores.npy`.

In [ ]:
!shimballm train --config colab_run.yaml --resume

## 6 · Recycle: re-anneal branches

The WSD schedule puts the whole LR decay inside Stage C, so the Stage-B stable checkpoint can be **re-annealed on different seeds/slices**. Each branch costs ~15% of a full run — that is the compute-recycling lever.

In [ ]:
!shimballm anneal --config colab_run.yaml --seed 1 --tag colab-seed1
!shimballm anneal --config colab_run.yaml --seed 2 --tag colab-seed2

import json, glob
for f in sorted(glob.glob(f'{OUT}/branches/*/summary.json')):
    s = json.load(open(f))
    print(f"{f.split('/')[-2]:>14}  val_ce={s['val']['student_ce']:.4f}")

## 7 · Training curves and run digest

In [ ]:
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(f'{OUT}/log.jsonl')]
STAGE_COLORS = {'A': 'tab:blue', 'B': 'tab:orange', 'C': 'tab:green'}

fig, ax = plt.subplots(2, 2, figsize=(12, 7))

# train loss
for s in sorted({r['stage'] for r in rows}):
    pts = [r for r in rows if r['stage'] == s and 'loss' in r]
    if pts:
        ax[0, 0].plot([r['tokens_total'] / 1e6 for r in pts],
                      [r['loss'] for r in pts], '.', ms=3,
                      color=STAGE_COLORS.get(s[0]), label=f'Stage {s[0]}')
ax[0, 0].set(title='Training loss', xlabel='tokens (M)', ylabel='CE')
ax[0, 0].legend(markerscale=4)

# validation CE
pts = [r for r in rows if 'val_student_ce' in r]
ax[0, 1].plot([r['tokens_total'] / 1e6 for r in pts],
              [r['val_student_ce'] for r in pts], 'o-', ms=4)
ax[0, 1].set(title='Validation CE (gates fire on its slope)',
             xlabel='tokens (M)', ylabel='CE')

# learning rate + context ramp
pts = [r for r in rows if 'lr' in r]
ax[1, 0].plot([r['tokens_total'] / 1e6 for r in pts],
              [r['lr'] for r in pts], '-', lw=1)
ax[1, 0].set(title='WSD learning rate', xlabel='tokens (M)', ylabel='lr')

ax[1, 1].plot([r['tokens_total'] / 1e6 for r in pts],
              [r['ctx'] for r in pts], '-', lw=1, color='tab:red')
ax[1, 1].set(title='Progressive context (2K → 4K)', xlabel='tokens (M)',
             ylabel='ctx')

fig.tight_layout()
plt.show()

In [ ]:
s = json.load(open(f'{OUT}/summary.json'))

print('=== run digest ' + '=' * 46)
print(f"model params      : {s['model_params']/1e6:8.1f} M")
print(f"tokens trained    : {s['tokens_total']:12,}")
print(f"scoring overhead  : {s['phase0']['flops'] / s['flops']['flops_student'] * 100:7.1f} % of student FLOPs")
print(f"FLOPs vs baseline : {100*s['flops_vs_baseline']:7.1f} %  (uniform training = 100%)")
print(f"final val CE      : {s['val']['student_ce']:.4f}")
print()
print('stage log: ' + '-' * 50)
for st in s['stage_log']:
    print(' ', st)

## 8 · Publish to the Hugging Face Hub (org `ShimbaLLM`)

Uploads the weights plus `config.yaml` + `summary.json`, and generates a 
model card with the full CROP recipe table and this run's measured results.
Create a write token at https://huggingface.co/settings/tokens first.

In [ ]:
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('HF write token: ')

# !shimballm push --config colab_run.yaml --repo ShimbaLLM/crop-smollm2-135m-colab
print('uncomment the push line once your token is set')

## Next steps

* **Scale up** — raise `total_tokens` in `colab_run.yaml` and `--resume`; the projection tells you the FLOPs cost before you commit.
* **Ablate** — the paper's baselines are one-flag switches: `crop.enabled: false` (B1 uniform), `rho_start=rho_end=0` (B2 gating only), `doc_drop_quantile: 0` (B3 masking only).
* **GitHub** — push this repo (`git remote add origin … && git push -u origin main`), then anyone can install it with `pip install git+https://github.com/ShimbaLLM/shimballm.git`.
* **Cite** — see `README.md` for the BibTeX entry.